# 04 · Severidad de incendio con NBR y dNBR

**Objetivo:** Comparar condiciones pre y postevento para identificar áreas quemadas.

**Datos:** Sentinel-2 SR: B8 y B12.

**Relevancia para política ambiental y social:** Apoya respuesta, restauración y priorización de inspecciones.

**Limitaciones:** Las fechas deben ajustarse al evento real y verificarse con información de campo.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
pre = s2_composite("2025-06-01","2025-08-31")
post = s2_composite("2025-09-01","2025-11-30")
nbr_pre = pre.normalizedDifference(["B8","B12"])
nbr_post = post.normalizedDifference(["B8","B12"])
dnbr = nbr_pre.subtract(nbr_post).rename("dNBR")

Map.addLayer(dnbr, {"min":-0.2,"max":0.8,"palette":["green","white","orange","red"]}, "dNBR")
Map
